This script is used by LH to determine based of multiple criteria if someone is a reporter.

It is in the poc as an exploration of testing stored procedures and deploying them

It seems there could be a database for read using cdc for sync lh data -> but cdc makes dbx appropriate sort of i think 

cdc stream for gating with isreporter but reports themselves surely could be batched daily or some other slow process

dlt faster so for sync stuff may be what needed

163598 is a test userid

# Original (modified a bit :) )
```
/*
For a given user, are they a reporter
*/
CREATE OR REPLACE PROCEDURE databricks_poc2_ws_uks.tel_unified_reporting_gold.sp_IsReporter

(
    IN par_userId INT
)
LANGUAGE SQL
SQL SECURITY INVOKER 
AS 
BEGIN

  SELECT DISTINCT
  CASE WHEN 
    (CASE WHEN UAL.UserBK IS NULL THEN 0 ELSE 1 END   +
    CASE WHEN UGR.userId IS NULL THEN 0 ELSE 1 END   +
    CASE WHEN URU.UserBK IS NULL THEN 0 ELSE 1 END) = 0 THEN 0 ELSE 1 END AS IsReporter

  FROM databricks_poc2_ws_uks.tel_unified_reporting_bronze_auth.user AU
  -- Location based permission
  LEFT JOIN databricks_poc2_ws_uks.tel_unified_reporting_bronze_auth.useradminlocation UAL 
      ON AU.UserHK = UAL.Userhk
      AND UAL.Deleted = 'false'
      AND UAL.LocationBK NOT IN (1,190822) -- Unknown or Unselected
  -- User group based permission
  LEFT JOIN databricks_poc2_ws_uks.tel_unified_reporting_bronze_auth.usergroupreportertbl UGR 
      ON UGR.userId = AU.UserBK 
      AND NOT UGR.deleted
  -- User based permission
  LEFT JOIN databricks_poc2_ws_uks.tel_unified_reporting_bronze_auth.userreportinguser URU
    ON URU.ReportingUserHK = AU.UserHK
      AND URU.Deleted = 'false'

  WHERE AU.UserBK = par_userId --:Reporter
  AND AU.UserDeleted = 'false'

  ;
END;


CALL databricks_poc2_ws_uks.tel_unified_reporting_gold.sp_IsReporter
    (
      :userId
    );
/**/
```

Instead of sql derived table or view then logic done once. View makes it quicker. Then hit it with "trivial" sql.

In [0]:
%sql

CREATE OR REPLACE TABLE databricks_poc2_ws_uks.tel_unified_reporting_gold.user_reporter_status AS
SELECT
  AU.UserBK              AS user_id,
  CASE
    WHEN
      (CASE WHEN UAL.UserBK IS NULL THEN 0 ELSE 1 END +
       CASE WHEN UGR.userId IS NULL THEN 0 ELSE 1 END +
       CASE WHEN URU.UserBK IS NULL THEN 0 ELSE 1 END) = 0
    THEN 0
    ELSE 1
  END AS is_reporter,
  current_timestamp() AS last_calculated
FROM databricks_poc2_ws_uks.tel_unified_reporting_bronze_auth.user AU
LEFT JOIN databricks_poc2_ws_uks.tel_unified_reporting_bronze_auth.useradminlocation UAL
  ON AU.UserHK = UAL.Userhk
  AND UAL.Deleted = 'false'
  AND UAL.LocationBK NOT IN (1,190822)
LEFT JOIN databricks_poc2_ws_uks.tel_unified_reporting_bronze_auth.usergroupreportertbl UGR
  ON UGR.userId = AU.UserBK
  AND NOT UGR.deleted
LEFT JOIN databricks_poc2_ws_uks.tel_unified_reporting_bronze_auth.userreportinguser URU
  ON URU.ReportingUserHK = AU.UserHK
  AND URU.Deleted = 'false'
WHERE AU.UserDeleted = 'false';

paramatising

In [0]:
%sql
/*
For a given user, are they a reporter
*/
CREATE OR REPLACE PROCEDURE :catalog.:schema.sp_IsReporter
(
    IN par_userId INT
)
LANGUAGE SQL
SQL SECURITY INVOKER 
AS 
BEGIN

  SELECT DISTINCT
  CASE WHEN 
    (CASE WHEN UAL.UserBK IS NULL THEN 0 ELSE 1 END   +
    CASE WHEN UGR.userId IS NULL THEN 0 ELSE 1 END   +
    CASE WHEN URU.UserBK IS NULL THEN 0 ELSE 1 END) = 0 THEN 0 ELSE 1 END AS IsReporter

  FROM :catalog.:bronze_schema.user AU
  -- Location based permission
  LEFT JOIN :catalog.:bronze_schema.useradminlocation UAL 
      ON AU.UserHK = UAL.Userhk
      AND UAL.Deleted = 'false'
      AND UAL.LocationBK NOT IN (1,190822)
  -- User group based permission
  LEFT JOIN :catalog.:bronze_schema.usergroupreportertbl UGR 
      ON UGR.userId = AU.UserBK 
      AND NOT UGR.deleted
  -- User based permission
  LEFT JOIN :catalog.:bronze_schema.userreportinguser URU
    ON URU.ReportingUserHK = AU.UserHK
      AND URU.Deleted = 'false'

  WHERE AU.UserBK = par_userId
  AND AU.UserDeleted = 'false'
  ;
END;
